# 7. Apply the best method to all judgments

Uses the method with the highest dev macro F1 in `06_method_comparison`, or the one named in `METHOD`.

**Reads:** `intermediate/sentences.pkl`, `predictions/`, `method_comparison.csv`, and the corpus for outcomes.
**Writes:** `case_level_categories.csv`, one row per judgment.

**Before interpreting the outcome table, check the outcome labels.** "Petition dismissed" in a quashing petition means the defamation case continues (as in *Sanwaria*), which is the opposite of the complaint being dismissed.

## Setup

In [1]:
from argmine_common import *
from scipy.stats import chi2_contingency, fisher_exact

OUTCOME_COL = "outcome"
METHOD = "M1 revised keywords"   # None = best dev macro F1 from 06; M1's dev score is optimistic (its rules were written from dev sentences)

sents = load_sentences()
corpus = load_corpus()

## Categories per judgment

In [2]:
preds = load_preds()
if METHOD is None:
    res = pd.read_csv(OUT_DIR / "method_comparison.csv")
    METHOD = res[res.split == "dev"].set_index("method").macro_F1.idxmax()
ALL_PRED = preds[METHOD]["all"]

case = (pd.DataFrame(ALL_PRED, columns=[SHORT[c] for c in CATS])
        .assign(doc_id=sents.doc_id.values).groupby("doc_id").max())
case = case.join(corpus.set_index("doc_id")[[OUTCOME_COL]])
case.to_csv(OUT_DIR / "case_level_categories.csv")

print(f"Method: {METHOD}. Share of judgments relying on each ground:")
print(case[[SHORT[c] for c in CATS]].mean().round(3).to_string())
print("\nBy outcome (share of judgments in each outcome group):")
by_outcome = case.groupby(OUTCOME_COL)[[SHORT[c] for c in CATS]].mean().round(3)
by_outcome.insert(0, "n", case[OUTCOME_COL].value_counts())
by_outcome

Method: M1 revised keywords. Share of judgments relying on each ground:
Ingredients/exceptions    0.556
Speech vs reputation      0.273
Procedural                0.475
Insufficient evidence     0.232
Abuse of process          0.333

By outcome (share of judgments in each outcome group):


,n,Ingredients/exceptions,Speech vs reputation,Procedural,Insufficient evidence,Abuse of process
outcome,,,,,,
Convicted,13,0.769,0.231,0.769,0.615,0.154
Ongoing/Unknown,87,0.517,0.276,0.586,0.195,0.287
Quashed/Dismissed,197,0.558,0.274,0.406,0.223,0.365


## Outcome tests

In [3]:
# Does each ground appear more often in one outcome group? Chi-square across groups, Fisher for 2x2.
test_rows = []
groups = case[OUTCOME_COL].dropna()
for c in CATS:
    col = SHORT[c]
    tab = pd.crosstab(case.loc[groups.index, OUTCOME_COL], case.loc[groups.index, col])
    if tab.shape[1] < 2 or tab.shape[0] < 2:
        continue
    if tab.shape == (2, 2):
        _, p = fisher_exact(tab.values); test = "Fisher"
    else:
        _, p, _, _ = chi2_contingency(tab.values); test = "Chi-square"
    test_rows.append({"category": col, "test": test, "p_value": round(p, 4)})
pd.DataFrame(test_rows)

,category,test,p_value
0,Ingredients/exceptions,Chi-square,0.2314
1,Speech vs reputation,Chi-square,0.9410
2,Procedural,Chi-square,0.0019
3,Insufficient evidence,Chi-square,0.0033
4,Abuse of process,Chi-square,0.1630
